# Julia notebook to compute the symbolic quadratures for the solution of the robust diagnostic PG problem
twnh 

Aug '26.  

This notebook reads the output from `BuoyancyEquationSolution.ipynb` and `FrictionalGeostrophicEquationSolution.ipynb`, namely the Green's functions $G_{\mathfrak{u}}$ and $G_b$ and computes integrals using them. It then writes out the following expression for later use.

\begin{align}
G_{ \mathfrak{u}, b} (z; \xi') & = 
\int_{-H}^{0} G_{ \mathfrak{u}} (z; \xi) \, \int_{\xi}^{0}  G_b(z'; \xi') \; dz'  \; d \xi .
\end{align}

See `SymbolicsTesting.jl` to test this result against numerical quadrature.

The notebook makes a plot of this $G_{ \mathfrak{u}, b} (z; \xi')$ function.

The notebook also computes 

\begin{align}
G_{ \mathfrak{u}, {b_\xi}} (z) & = \int_{-H}^{0} G_{ \mathfrak{u}} (z; \xi) \, \int_{\xi}^{0} \partial_\xi G_b(z';\xi) |_{\xi = 0}   \; dz'  \; d \xi ,\\
G_{ \mathfrak{u}_y, b}& = \int_{-H}^{0} \partial_y G_{ \mathfrak{u}} (z; \xi) \, \int_{\xi}^{0}  G_b(z';\xi)   \; dz'  \; d \xi , \\
G_{ \mathfrak{u}_y, {b_\xi}} (z) & = \int_{-H}^{0} \partial_y G_{ \mathfrak{u}} (z; \xi) \, \int_{\xi}^{0} \partial_\xi G_b(z';\xi) |_{\xi = 0}   \; dz'  \; d \xi .
\end{align}

In [1]:
using SymPy
using Symbolics
using JLD2
using Infiltrator

In [ ]:
# Geometry symbolic parameters:
z,  ξ  = symbols("z  ξ",   real=true, negative=true) # Vertical coordinate and source location (both in [-H,0])
zp, ξp = symbols("zp ξp",  real=true, negative=true) # Vertical coordinate and source location (both in [-H,0])
x, y   = symbols("x y",    real=true)                # Horizontal coordinates
H      = symbols("H",      real=true, positive=true) # Domain depth

ψ      = symbols("ψ",      real=true, positive=true) 
φ      = symbols("φ", imaginary=true)

κ      = symbols("κ",      real=true, positive=true) # Diffusivity parameter
ν      = symbols("ν",      real=true, positive=true) # Viscosity parameters

Ek     = symbols("Ek",     real=true, positive=true) # Ekman number
f      = symbols("f",      real=true)                # Coriolis parameter
β      = symbols("β",      real=true, positive=true) # Planetary vorticity gradient

# Dictionary for parsing strings.
locals = Dict(
    "x" => x,
    "y" => y,
    "z" => z,
    "ξ" => ξ,
    "κ" => κ,
    "ν" => ν,
    "φ" => φ,
    "ψ" => ψ,
    "H" => H
    )

# Parse expressions using the canonical symbols.
Guv_str = read("sympy_Guv_expression.txt", String)
Guv = sympy.sympify(Guv_str, locals=locals)
println("0.1 Guv expression loaded from file:")
display(Guv)
println()

Gb_str  = read("sympy_Gb_expression.txt", String)
Gb  = sympy.sympify(Gb_str,  locals=locals)
println("0.2 Gb expression loaded from file:")
display(Gb)
println()

In [ ]:
# ---------------------------------------------------------------------
# Faster symbolic utilities
# ---------------------------------------------------------------------

function contains_symbol(expr, var)
    try
        return var in expr.free_symbols
    catch
        return occursin(string(var), string(expr))
    end
end

function has_unevaluated_integral(expr)
    s = string(expr)
    return occursin("Integral", s)
end

function bad_integral_result(expr, var)
    return contains_symbol(expr, var) ||
           has_unevaluated_integral(expr) ||
           occursin("Piecewise", string(expr))
end

function light_simplify(expr)
    expr = sympy.powsimp(expr, force=false)
    expr = sympy.factor_terms(expr)
    expr = sympy.cancel(expr)
    return expr
end

function medium_simplify(expr)
    expr = sympy.together(expr)
    expr = sympy.cancel(expr)
    expr = sympy.factor_terms(expr)
    expr = sympy.powsimp(expr, force=false)
    return expr
end

const integral_cache = Dict{Tuple{String,String,String,String}, Any}()
const dep_integral_cache = Dict{Tuple{String,String,String,String}, Any}()

function sympy_is_add(expr)
    try
        return Bool(expr.is_Add)
    catch
        return occursin("Add", string(expr.func))
    end
end

function additive_args(expr)
    if sympy_is_add(expr)
        return collect(expr.args)
    else
        return Any[expr]
    end
end

function top_level_terms(expr; do_expand=true, split_rational_numerator=true)
    ex = do_expand ? sympy.expand_mul(expr) : expr

    # First try ordinary additive splitting.
    ex_expanded = do_expand ? sympy.expand(ex) : ex
    terms = additive_args(ex_expanded)

    if length(terms) > 1
        return terms
    end

    if !split_rational_numerator
        return terms
    end

    # Fallback:
    # If expr is one huge rational expression, split the numerator.
    num, den = sympy.fraction(ex_expanded)

    num_expanded = sympy.expand(num)
    num_terms = additive_args(num_expanded)

    if length(num_terms) > 1
        println("    split rational numerator into $(length(num_terms)) terms")
        return Any[t / den for t in num_terms]
    end

    # Second fallback: use together() form, then split numerator.
    ex2 = sympy.together(ex)
    num2, den2 = sympy.fraction(ex2)
    num2 = sympy.expand(num2)
    num2_terms = additive_args(num2)

    if length(num2_terms) > 1
        println("    split together() numerator into $(length(num2_terms)) terms")
        return Any[t / den2 for t in num2_terms]
    end

    return Any[ex_expanded]
end

function normalize_for_integration(expr)
    # Stronger than the per-term powsimp, but applied only to one small term.
    expr = sympy.powsimp(expr, force=true)
    expr = sympy.cancel(expr)
    expr = sympy.factor_terms(expr)
    expr = sympy.powsimp(expr, force=true)
    return expr
end

function independent_dependent_parts(t, var)
    try
        # First normalize/cancel exponentials like
        # exp(-2ξψ)/(exp(ξφ)*exp(-ξψ)) -> exp(-ξφ - ξψ)
        tt = normalize_for_integration(t)

        parts = tt.as_independent(var, as_Add=false)
        indep = parts[1]
        dep   = parts[2]

        dep = sympy.powsimp(dep, force=true)

        return indep, dep
    catch
        return sympy.S(1), t
    end
end

function is_sympy_exp(expr)
    try
        return string(expr.func) == "exp"
    catch
        return false
    end
end

function combine_exp_products(expr)
    try
        expr = sympy.powsimp(expr, force=true)
        expr = sympy.expand_power_exp(expr)
        expr = sympy.powsimp(expr, force=true)
        return expr
    catch
        return expr
    end
end

function try_integrate_simple_exp(dep, var, a, b)
    dep = combine_exp_products(dep)

    # Case 1:
    #   dep = exp(linear(var))
    try
        if is_sympy_exp(dep)
            arg = dep.args[1]
            λ = sympy.diff(arg, var)

            if !contains_symbol(λ, var)
                return (dep.subs(var, b) - dep.subs(var, a)) / λ
            end
        end
    catch
    end

    # Case 2:
    #   dep = product of exponentials, e.g.
    #   exp(-ξ*φ)*exp(-ξ*ψ)
    #
    # Try to force it into exp(linear expression).
    try
        dep2 = sympy.powsimp(dep, force=true)

        if is_sympy_exp(dep2)
            arg = dep2.args[1]
            λ = sympy.diff(arg, var)

            if !contains_symbol(λ, var)
                return (dep2.subs(var, b) - dep2.subs(var, a)) / λ
            end
        end
    catch
    end

    # Case 3:
    #   dep = polynomial(var) * exp(linear(var))
    try
        dep2 = sympy.powsimp(dep, force=true)
        factors = collect(dep2.as_ordered_factors())

        exp_factor = nothing
        other = sympy.S(1)

        for fac in factors
            fac2 = sympy.powsimp(fac, force=true)

            if is_sympy_exp(fac2)
                if exp_factor === nothing
                    exp_factor = fac2
                else
                    exp_factor = sympy.powsimp(exp_factor * fac2, force=true)
                end
            else
                other *= fac2
            end
        end

        if exp_factor !== nothing
            exp_factor = sympy.powsimp(exp_factor, force=true)

            if is_sympy_exp(exp_factor)
                arg = exp_factor.args[1]
                λ = sympy.diff(arg, var)

                if contains_symbol(λ, var)
                    return nothing
                end

                poly = sympy.Poly(other, var)
                deg = Int(poly.degree())

                if deg <= 2
                    F = sympy.integrate(other * exp_factor, var, conds="none")
                    return F.subs(var, b) - F.subs(var, a)
                end
            end
        end
    catch
    end

    return nothing
end

function integrate_dep_cached(dep, var, a, b)
    dep = combine_exp_products(dep)

    key = (string(dep), string(var), string(a), string(b))

    if haskey(dep_integral_cache, key)
        return dep_integral_cache[key]
    end

    # If dep is actually independent of var.
    if !contains_symbol(dep, var)
        res = dep * (b - a)
        dep_integral_cache[key] = res
        return res
    end

    # Manual exponential cases.
    manual = try_integrate_simple_exp(dep, var, a, b)
    if manual !== nothing
        dep_integral_cache[key] = manual
        return manual
    end

    # Try a few tiny-core variants.
    candidates = (
        dep,
        sympy.powsimp(dep, force=true),
        sympy.expand_power_exp(dep),
        sympy.powsimp(sympy.expand_power_exp(dep), force=true),
        sympy.cancel(sympy.powsimp(dep, force=true)),
    )

    for dep_c in candidates
        dep_c = combine_exp_products(dep_c)

        manual = try_integrate_simple_exp(dep_c, var, a, b)
        if manual !== nothing
            dep_integral_cache[key] = manual
            return manual
        end

        res = sympy.integrate(dep_c, (var, a, b), conds="none")

        if !bad_integral_result(res, var)
            dep_integral_cache[key] = res
            return res
        end

        F = sympy.integrate(dep_c, var, conds="none")
        res = F.subs(var, b) - F.subs(var, a)

        if !bad_integral_result(res, var)
            dep_integral_cache[key] = res
            return res
        end
    end

    println("WARNING: integrate_dep_cached failed.")
    println("  dep = ", dep)
    println("  combined dep = ", combine_exp_products(dep))

    res = sympy.integrate(dep, (var, a, b), conds="none")
    dep_integral_cache[key] = res
    return res
end

function integrate_term_cached(t, var, a, b)
    # Normalize full term first. This is important for cancelling exponentials
    # between numerator and denominator.
    t_norm = normalize_for_integration(t)

    key = (string(t_norm), string(var), string(a), string(b))

    if haskey(integral_cache, key)
        return integral_cache[key]
    end

    indep, dep = independent_dependent_parts(t_norm, var)

    dep = combine_exp_products(dep)

    dep_int = integrate_dep_cached(dep, var, a, b)
    res = indep * dep_int

    integral_cache[key] = res
    return res
end

function diagnose_integrand(expr, var; name="expr")
    println()
    println("Diagnostics for $name")
    println("  full func:          ", expr.func)
    println("  full is_Add:        ", sympy_is_add(expr))
    println("  full args length:   ", length(expr.args))
    println("  full string length: ", length(string(expr)))

    ex = sympy.expand_mul(expr)
    ex = sympy.expand(ex)

    println("  expanded func:          ", ex.func)
    println("  expanded is_Add:        ", sympy_is_add(ex))
    println("  expanded args length:   ", length(ex.args))
    println("  expanded string length: ", length(string(ex)))

    num, den = sympy.fraction(ex)
    num_exp = sympy.expand(num)
    den_exp = sympy.expand(den)

    println("  numerator func:          ", num_exp.func)
    println("  numerator is_Add:        ", sympy_is_add(num_exp))
    println("  numerator args length:   ", length(num_exp.args))
    println("  numerator string length: ", length(string(num_exp)))
    println("  denominator contains $var: ", contains_symbol(den_exp, var))
    println("  denominator string length: ", length(string(den_exp)))

    num_terms = additive_args(num_exp)
    println("  numerator additive terms: ", length(num_terms))
    println()
end

function integrate_terms_fast(expr, integ_tuple;
                              simplify_input=false,
                              compress_every=10_000,
                              max_single_term_chars=20_000,
                              print_each_term=true)

    var, a, b = integ_tuple

    if simplify_input
        expr = light_simplify(expr)
    end

    terms = top_level_terms(expr; do_expand=true, split_rational_numerator=true)

    res = sympy.S(0)

    nterms = length(terms)
    println("    integrating $nterms terms from $a to $b ...")

    if nterms == 1
        term_size = length(string(terms[1]))

        println("    WARNING: only one top-level term found.")
        println("    term head = ", terms[1].func)
        println("    term size = ", term_size)

        if term_size > max_single_term_chars
            diagnose_integrand(expr, var; name="failed one-term integrand")
            error("Refusing to integrate one huge symbolic term of size $term_size.")
        end
    end

    for (i, t) in enumerate(terms)
        if print_each_term
            println("      term $i / $nterms, chars = ", length(string(t)))
        end

        # Very light preprocessing only.
        t_s = sympy.powsimp(t, force=false)

        indep, dep = independent_dependent_parts(t_s, var)

        if print_each_term
            println("        indep chars = ", length(string(indep)),
                ", dep chars = ", length(string(dep)),
                ", dep = ", dep)
        end

        elapsed = @elapsed begin
            ti = integrate_term_cached(t_s, var, a, b)
            res += ti
        end

        if print_each_term
            println("        done in ", round(elapsed, digits=4), " s")
        elseif i % 10 == 0
            println("      integrated $i / $nterms")
        end

        if bad_integral_result(ti, var)
            println("WARNING: bad term integral at term $i / $nterms")
            println("  term   = ", t_s)
            println("  result = ", ti)
            @infiltrate
        end

        if i % compress_every == 0
            # Mild compression only.
            res = sympy.factor_terms(res)
            res = sympy.powsimp(res, force=false)
            println("      compressed accumulated result at $i / $nterms")
        end
    end

    println("    simplifying accumulated integral...")
    res = medium_simplify(res)

    if bad_integral_result(res, var)
        println("WARNING: final integrated result still contains $var or bad structure.")
        @infiltrate
    end

    return res
end

function diagnose_integrand(expr, var; name="expr")
    println()
    println("Diagnostics for $name")
    println("  full func:          ", expr.func)
    println("  full is_Add:        ", sympy_is_add(expr))
    println("  full args length:   ", length(expr.args))
    println("  full string length: ", length(string(expr)))

    ex = sympy.expand_mul(expr)
    ex = sympy.expand(ex)

    println("  expanded func:          ", ex.func)
    println("  expanded is_Add:        ", sympy_is_add(ex))
    println("  expanded args length:   ", length(ex.args))
    println("  expanded string length: ", length(string(ex)))

    num, den = sympy.fraction(ex)
    num_exp = sympy.expand(num)
    den_exp = sympy.expand(den)

    println("  numerator func:          ", num_exp.func)
    println("  numerator is_Add:        ", sympy_is_add(num_exp))
    println("  numerator args length:   ", length(num_exp.args))
    println("  numerator string length: ", length(string(num_exp)))
    println("  denominator contains $var: ", contains_symbol(den_exp, var))
    println("  denominator string length: ", length(string(den_exp)))

    num_terms = additive_args(num_exp)
    println("  numerator additive terms: ", length(num_terms))
    println()
end

#### Compute $ \int_{\xi}^{0}  G_b(z';\xi ')  \; dz' $

In [ ]:

println()
println("1.1 Symbolic computation of the integral of Gb...")
@time begin
    tmp_Gb = sympy.expand(Gb.subs(Dict(z => zp, ξ => ξp)))
    inner_Gb_integral = sympy.integrate(tmp_Gb, (zp, ξ, 0))
    max_obj = sympy.Max(ξ,ξp)
    min_obj = sympy.Min(0, max_obj)
    inner_Gb_integral = inner_Gb_integral.subs(min_obj,max_obj)
    # Create two explicit branches
    expr_if_xi_ge_xip = inner_Gb_integral.subs(sympy.Max(ξ, ξp), ξ)
    expr_if_xi_lt_xip = inner_Gb_integral.subs(sympy.Max(ξ, ξp),ξp)

    # Form a piecewise object
    inner_Gb_integral = sympy.Piecewise(
    (expr_if_xi_ge_xip, sympy.Ge(ξ, ξp)),
    (expr_if_xi_lt_xip, sympy.Lt(ξ, ξp)))
end

println("1.1 Done.")
# display(inner_Gb_integral)

#### Compute `Guvb` $G_{ \mathfrak{u}, b} = \int_{-H}^{0} G_{ \mathfrak{u}} (z; \xi) \, \int_{\xi}^{0} G_b(z';\xi)  \; dz'  \; d \xi $

In [ ]:
function integrate_Guv_Gb_fast(Guv_branch1, Guv_branch2, inner_Gb_integral;
                               print_each_term=false,
                               compress_every=10_000)

    println("  integrate_Guv_Gb_fast: Starting integration of Guv * Gb.")

    # ------------------------------------------------------------------
    # Extract branch expressions
    #
    # Guv:
    #   Guv_branch1: z ≤ ξ  ⇔ ξ ≥ z
    #   Guv_branch2: z ≥ ξ  ⇔ ξ ≤ z
    #
    # inner_Gb_integral:
    #   Gbint_branch1: ξ ≥ ξp
    #   Gbint_branch2: ξ <  ξp
    # ------------------------------------------------------------------
    Gbint_branch1 = inner_Gb_integral.args[1].args[1]   # ξ ≥ ξp
    Gbint_branch2 = inner_Gb_integral.args[2].args[1]   # ξ <  ξp

    println("  integrate_Guv_Gb_fast: Extracted Gb integral branches.")

    # ------------------------------------------------------------------
    # Lightly simplify individual branches only.
    # Do NOT simplify the full products globally.
    # ------------------------------------------------------------------
    Guv_branch1_s   = light_simplify(Guv_branch1)
    Guv_branch2_s   = light_simplify(Guv_branch2)
    Gbint_branch1_s = light_simplify(Gbint_branch1)
    Gbint_branch2_s = light_simplify(Gbint_branch2)

    println("  integrate_Guv_Gb_fast: Light-simplified individual branches.")

    # ------------------------------------------------------------------
    # Products
    #
    # f11: ξ ≥ z,  ξ ≥ ξp
    # f12: ξ ≥ z,  ξ <  ξp
    # f21: ξ ≤ z,  ξ ≥ ξp
    # f22: ξ ≤ z,  ξ <  ξp
    #
    # Important:
    #   Use expand_mul, not simplify.
    #   simplify/cancel of the full product can recombine terms and make
    #   integration much harder.
    # ------------------------------------------------------------------
    f11 = sympy.expand_mul(Guv_branch1_s * Gbint_branch1_s)
    f12 = sympy.expand_mul(Guv_branch1_s * Gbint_branch2_s)
    f21 = sympy.expand_mul(Guv_branch2_s * Gbint_branch1_s)
    f22 = sympy.expand_mul(Guv_branch2_s * Gbint_branch2_s)

    println("  integrate_Guv_Gb_fast: Prepared product branches.")

    n11 = length(top_level_terms(f11; do_expand=true, split_rational_numerator=true))
    n12 = length(top_level_terms(f12; do_expand=true, split_rational_numerator=true))
    n21 = length(top_level_terms(f21; do_expand=true, split_rational_numerator=true))
    n22 = length(top_level_terms(f22; do_expand=true, split_rational_numerator=true))

    println("    f11 terms: ", n11)
    println("    f12 terms: ", n12)
    println("    f21 terms: ", n21)
    println("    f22 terms: ", n22)

    # ------------------------------------------------------------------
    # Case A: z ≤ ξp
    #
    # Ordering:
    #
    #   -H ≤ ... ≤ z ≤ ξp ≤ ... ≤ 0
    #
    # Therefore:
    #
    #   ξ ∈ [-H, z]     → ξ ≤ z and ξ < ξp   → f22
    #   ξ ∈ [z, ξp]     → ξ ≥ z and ξ < ξp   → f12
    #   ξ ∈ [ξp, 0]     → ξ ≥ z and ξ ≥ ξp   → f11
    # ------------------------------------------------------------------
    println("  integrate_Guv_Gb_fast: Case A, z ≤ ξp")

    I_A_22 = integrate_terms_fast(f22, (ξ, -H, z);
                                  simplify_input=false,
                                  compress_every=compress_every,
                                  print_each_term=print_each_term)

    println("    Computed I_A_22 = ∫_{-H}^{z} f22 dξ")

    I_A_12 = integrate_terms_fast(f12, (ξ, z, ξp);
                                  simplify_input=false,
                                  compress_every=compress_every,
                                  print_each_term=print_each_term)

    println("    Computed I_A_12 = ∫_{z}^{ξp} f12 dξ")

    I_A_11 = integrate_terms_fast(f11, (ξ, ξp, 0);
                                  simplify_input=false,
                                  compress_every=compress_every,
                                  print_each_term=print_each_term)

    println("    Computed I_A_11 = ∫_{ξp}^{0} f11 dξ")

    println("    Simplifying Case A result...")
    I_A = medium_simplify(I_A_22 + I_A_12 + I_A_11)

    # ------------------------------------------------------------------
    # Case B: ξp < z
    #
    # Ordering:
    #
    #   -H ≤ ... ≤ ξp < z ≤ ... ≤ 0
    #
    # Therefore:
    #
    #   ξ ∈ [-H, ξp]    → ξ < ξp and ξ ≤ z   → f22
    #   ξ ∈ [ξp, z]     → ξ ≥ ξp and ξ ≤ z   → f21
    #   ξ ∈ [z, 0]      → ξ ≥ ξp and ξ ≥ z   → f11
    # ------------------------------------------------------------------
    println("  integrate_Guv_Gb_fast: Case B, ξp < z")

    I_B_22 = integrate_terms_fast(f22, (ξ, -H, ξp);
                                  simplify_input=false,
                                  compress_every=compress_every,
                                  print_each_term=print_each_term)

    println("    Computed I_B_22 = ∫_{-H}^{ξp} f22 dξ")

    I_B_21 = integrate_terms_fast(f21, (ξ, ξp, z);
                                  simplify_input=false,
                                  compress_every=compress_every,
                                  print_each_term=print_each_term)

    println("    Computed I_B_21 = ∫_{ξp}^{z} f21 dξ")

    I_B_11 = integrate_terms_fast(f11, (ξ, z, 0);
                                  simplify_input=false,
                                  compress_every=compress_every,
                                  print_each_term=print_each_term)

    println("    Computed I_B_11 = ∫_{z}^{0} f11 dξ")

    println("    Simplifying Case B result...")
    I_B = medium_simplify(I_B_22 + I_B_21 + I_B_11)

    # ------------------------------------------------------------------
    # Final Piecewise
    # ------------------------------------------------------------------
    println("  integrate_Guv_Gb_fast: Building Piecewise result.")

    Guvb = sympy.Piecewise(
        (I_A, sympy.Le(z, ξp)),
        (I_B, sympy.Gt(z, ξp))
    )

    return Guvb
end

In [ ]:
println()
println("1.2 Symbolic computation of Guvb...")
@time begin
    empty!(integral_cache)
    empty!(dep_integral_cache)

    Guv_branch1 = Guv.args[1].args[1]   # z ≤ ξ
    Guv_branch2 = Guv.args[2].args[1]   # z ≥ ξ

    Guvb = integrate_Guv_Gb_fast(
        Guv_branch1,
        Guv_branch2,
        inner_Gb_integral;
        print_each_term=false,
        compress_every=10_000,
    )
end

println("1.2 Done.")
# println("1.2 Skipped.")
# display(Guvb)

#### compute $ \int_{\xi}^{0} \partial_\xi G_b(z';\xi) |_{\xi = 0}   \; dz' $

In [ ]:
println()
println("1.3 Symbolic computation of integral of Gb_ξ...")
@time begin
    tmp_Gb2 = sympy.diff(Gb,ξ).subs(ξ,0).subs(z,zp)
    inner_Gb2_integral = sympy.integrate(tmp_Gb2, (zp, ξ, 0))
end

println("1.3 Done.")
# display(inner_Gb2_integral)

#### Now compute `Guvb_xi` $G_{ \mathfrak{u}, b_\xi} = \int_{-H}^{0} G_{ \mathfrak{u}} (z; \xi) \, \int_{\xi}^{0} \partial_\xi G_b(z';\xi) |_{\xi = 0}   \; dz'  \; d \xi $

In [ ]:
function compute_Guvb_xi_fast(Guv, inner_Gb2_integral;
                              print_each_term=false,
                              compress_every=10_000)

    println("  compute_Guvb_xi_fast: Starting.")

    # ------------------------------------------------------------------
    # Extract Guv branches
    #
    # Guv_branch1: z ≤ ξ  ⇔ ξ ≥ z
    # Guv_branch2: z ≥ ξ  ⇔ ξ ≤ z
    # ------------------------------------------------------------------
    Guv_branch1 = Guv.args[1].args[1]   # z ≤ ξ
    Guv_branch2 = Guv.args[2].args[1]   # z ≥ ξ

    println("  compute_Guvb_xi_fast: Extracted Guv branches.")

    # ------------------------------------------------------------------
    # Simplify individual factors only.
    # Do not simplify the full products globally.
    # ------------------------------------------------------------------
    Guv_branch1_s = light_simplify(Guv_branch1)
    Guv_branch2_s = light_simplify(Guv_branch2)

    inner_Gb2_integral_s = medium_simplify(inner_Gb2_integral)

    println("  compute_Guvb_xi_fast: Simplified individual factors.")

    # ------------------------------------------------------------------
    # Products
    #
    # For ξ ∈ [-H, z], use Guv_branch2.
    # For ξ ∈ [z, 0],  use Guv_branch1.
    # ------------------------------------------------------------------
    f_lower = sympy.expand_mul(Guv_branch2_s * inner_Gb2_integral_s)
    f_upper = sympy.expand_mul(Guv_branch1_s * inner_Gb2_integral_s)

    println("  compute_Guvb_xi_fast: Prepared product branches.")

    n_lower = length(top_level_terms(f_lower; do_expand=true, split_rational_numerator=true))
    n_upper = length(top_level_terms(f_upper; do_expand=true, split_rational_numerator=true))

    println("    lower branch terms: ", n_lower)
    println("    upper branch terms: ", n_upper)

    # Optional diagnostics if needed
    # diagnose_integrand(f_lower, ξ; name="Guvb_xi lower branch")
    # diagnose_integrand(f_upper, ξ; name="Guvb_xi upper branch")

    # ------------------------------------------------------------------
    # Integrate termwise
    # ------------------------------------------------------------------
    I_lower = integrate_terms_fast(f_lower, (ξ, -H, z);
                                   simplify_input=false,
                                   compress_every=compress_every,
                                   print_each_term=print_each_term)

    println("    Computed I_lower = ∫_{-H}^{z} Guv_branch2 * inner_Gb2_integral dξ")

    I_upper = integrate_terms_fast(f_upper, (ξ, z, 0);
                                   simplify_input=false,
                                   compress_every=compress_every,
                                   print_each_term=print_each_term)

    println("    Computed I_upper = ∫_{z}^{0} Guv_branch1 * inner_Gb2_integral dξ")

    # ------------------------------------------------------------------
    # Combine and simplify
    # ------------------------------------------------------------------
    println("  compute_Guvb_xi_fast: Combining and simplifying.")

    Guvb_xi = medium_simplify(I_lower + I_upper)

    # A final slightly stronger but still staged simplification.
    Guvb_xi = sympy.together(Guvb_xi)
    Guvb_xi = sympy.cancel(Guvb_xi)
    Guvb_xi = sympy.factor_terms(Guvb_xi)
    Guvb_xi = sympy.powsimp(Guvb_xi, force=false)

    println("  compute_Guvb_xi_fast: Done.")

    return Guvb_xi
end

In [ ]:
println()
println("1.4 Symbolic computation of Guvb_ξ...")
@time begin
    empty!(integral_cache)
    empty!(dep_integral_cache)

    Guvb_xi = compute_Guvb_xi_fast(
        Guv,
        inner_Gb2_integral;
        print_each_term=false,
        compress_every=10_000,
    )
end

println("1.4 Done.")
# println("1.4 Skipped.")
# display(Guvb_xi)

#### Compute $\partial_y G_{\mathfrak{u}}$

In [ ]:
println()
println("1.5 Symbolic computation of Guv_y...")
@time begin
    Guv_φ  = sympy.diff(Guv,φ)
    Guv_H  = sympy.diff(Guv,H)
    fy     = SymFunction("f",  real=true)                # Coriolis parameter
    H_y    = symbols("H_y",    real=true)                   # Depth gradient
    φ_term = symbols("φ_term", imaginary=true)
    φ_eqn  = sqrt(im * fy(y) / (ν * Ek))
    φ_derivative = diff(φ_eqn,y).subs(diff(fy(y),y),β)
    Latex_φ_derivative = β/ (2 * fy(y)) * φ_term
    @assert Latex_φ_derivative.subs(φ_term, φ_eqn) == φ_derivative
    φ_derivative = Latex_φ_derivative.subs(fy(y), f).subs(φ_term, φ)
    Guv_y = φ_derivative * Guv_φ + H_y * Guv_H
    Guv_y = sympy.simplify(Guv_y)
    end

println("1.5 Done and Latex formula for φ derivative passes.")
# display(Guv_y)

#### Now compute `Guv_y_b_xi` $G_{ \mathfrak{u}, b_\xi} = \int_{-H}^{0} \partial_y G_{ \mathfrak{u}} (z; \xi) \, \int_{\xi}^{0} \partial_\xi G_b(z';\xi) |_{\xi = 0}   \; dz'  \; d \xi $

In [ ]:
println()
println("1.6 Symbolic computation of Guv_y_b_xi...")
@time begin
    empty!(integral_cache)
    empty!(dep_integral_cache)

    inner_Gb2_integral_s = medium_simplify(inner_Gb2_integral)
    println("  Simplified inner_Gb2_integral.")

    Guvφ_branch1 = Guv_y.args[1].args[1]
    Guvφ_branch2 = Guv_y.args[2].args[1]

    # Simplify the individual Green's-function branches only.
    Guvφ_branch1 = light_simplify(Guvφ_branch1)
    Guvφ_branch2 = light_simplify(Guvφ_branch2)

    println("  Prepared Guv_y branches.")

    # Expand products into additive terms.
    # Do NOT cancel/factor the whole branch after this.
    branch1 = sympy.expand_mul(Guvφ_branch1 * inner_Gb2_integral_s)
    branch2 = sympy.expand_mul(Guvφ_branch2 * inner_Gb2_integral_s)

    n1 = length(top_level_terms(branch1; do_expand=true, split_rational_numerator=true))
    n2 = length(top_level_terms(branch2; do_expand=true, split_rational_numerator=true))

    println("  Prepared product branches.")
    println("    branch1 terms: ", n1)
    println("    branch2 terms: ", n2)

    # diagnose_integrand(branch1, ξ; name="branch1")
    # diagnose_integrand(branch2, ξ; name="branch2")

    I1 = integrate_terms_fast(branch1, (ξ, z, 0);
                              simplify_input=false,
                              compress_every=10_000,
                              print_each_term=false)

    println("  Computed I1 = ∫_{z}^{0} Guv_y * Gb_ξ dξ")

    I2 = integrate_terms_fast(branch2, (ξ, -H, z);
                              simplify_input=false,
                              compress_every=10_000,
                              print_each_term=false)

    println("  Computed I2 = ∫_{-H}^{z} Guv_y * Gb_ξ dξ")

    println("  Simplifying I1 and I2...")
    I1 = medium_simplify(I1)
    I2 = medium_simplify(I2)

    Guv_y_b_xi = medium_simplify(I1 + I2)

    println("  Combined I1 + I2.")

    println("  Simplifying final Guv_y_b_xi...")
    Guv_y_b_xi = sympy.together(Guv_y_b_xi)
    Guv_y_b_xi = sympy.cancel(Guv_y_b_xi)
    Guv_y_b_xi = sympy.factor_terms(Guv_y_b_xi)
    Guv_y_b_xi = sympy.powsimp(Guv_y_b_xi, force=false)

    println("  Final simplification done.")
end

@time begin
    Guv_y_b_xi_str = string(Guv_y_b_xi)

    println("  string length: ", length(Guv_y_b_xi_str))
    println("  contains unevaluated Integral: ", occursin("Integral", Guv_y_b_xi_str))
    println("  contains Piecewise:             ", occursin("Piecewise", Guv_y_b_xi_str))
    println("  contains zoo:                   ", occursin("zoo", Guv_y_b_xi_str))
    println("  contains nan:                   ", occursin("nan", lowercase(Guv_y_b_xi_str)))
end

println("1.6 Done.")


#### Now compute `Guv_y_b` $G_{ \mathfrak{u}_y, b} = \int_{-H}^{0} \partial_y G_{ \mathfrak{u}} (z; \xi) \, \int_{\xi}^{0}  G_b(z';\xi)  \; dz'  \; d \xi $

In [ ]:
function integrate_Guv_y_Gb_fast(Guv_y, inner_Gb_integral;
                                 print_each_term=false,
                                 compress_every=10_000)

    println("  integrate_Guv_y_Gb_fast: Starting integration of Guv_y * inner_Gb_integral.")

    # ------------------------------------------------------------------
    # Extract Guv_y branch expressions
    #
    # Guv_y:
    #   Guvφ_branch1: z ≤ ξ  ⇔ ξ ≥ z
    #   Guvφ_branch2: z ≥ ξ  ⇔ ξ ≤ z
    #
    # inner_Gb_integral:
    #   Gbint_branch1: ξ ≥ ξp
    #   Gbint_branch2: ξ <  ξp
    # ------------------------------------------------------------------
    Guvφ_branch1 = Guv_y.args[1].args[1]                # z ≤ ξ
    Guvφ_branch2 = Guv_y.args[2].args[1]                # z ≥ ξ

    Gbint_branch1 = inner_Gb_integral.args[1].args[1]   # ξ ≥ ξp
    Gbint_branch2 = inner_Gb_integral.args[2].args[1]   # ξ <  ξp

    println("  integrate_Guv_y_Gb_fast: Extracted branches.")

    println("    inner_Gb_integral branch 1 condition: ", inner_Gb_integral.args[1].args[2])
    println("    inner_Gb_integral branch 2 condition: ", inner_Gb_integral.args[2].args[2])

    # ------------------------------------------------------------------
    # Light simplification of individual branches only.
    #
    # Do NOT use full simplify on products.
    # ------------------------------------------------------------------
    Guvφ_branch1_s = light_simplify(Guvφ_branch1)
    Guvφ_branch2_s = light_simplify(Guvφ_branch2)

    Gbint_branch1_s = light_simplify(Gbint_branch1)
    Gbint_branch2_s = light_simplify(Gbint_branch2)

    println("  integrate_Guv_y_Gb_fast: Light-simplified individual branches.")

    # ------------------------------------------------------------------
    # Products
    #
    # f11: ξ ≥ z,  ξ ≥ ξp
    # f12: ξ ≥ z,  ξ <  ξp
    # f21: ξ ≤ z,  ξ ≥ ξp
    # f22: ξ ≤ z,  ξ <  ξp
    # ------------------------------------------------------------------
    f11 = sympy.expand_mul(Guvφ_branch1_s * Gbint_branch1_s)
    f12 = sympy.expand_mul(Guvφ_branch1_s * Gbint_branch2_s)
    f21 = sympy.expand_mul(Guvφ_branch2_s * Gbint_branch1_s)
    f22 = sympy.expand_mul(Guvφ_branch2_s * Gbint_branch2_s)

    println("  integrate_Guv_y_Gb_fast: Prepared product branches.")

    n11 = length(top_level_terms(f11; do_expand=true, split_rational_numerator=true))
    n12 = length(top_level_terms(f12; do_expand=true, split_rational_numerator=true))
    n21 = length(top_level_terms(f21; do_expand=true, split_rational_numerator=true))
    n22 = length(top_level_terms(f22; do_expand=true, split_rational_numerator=true))

    println("    f11 terms: ", n11)
    println("    f12 terms: ", n12)
    println("    f21 terms: ", n21)
    println("    f22 terms: ", n22)

    # Optional diagnostics if needed.
    # diagnose_integrand(f11, ξ; name="Guv_y_b f11")
    # diagnose_integrand(f12, ξ; name="Guv_y_b f12")
    # diagnose_integrand(f21, ξ; name="Guv_y_b f21")
    # diagnose_integrand(f22, ξ; name="Guv_y_b f22")

    # ------------------------------------------------------------------
    # Case A: z ≤ ξp
    #
    # Ordering:
    #
    #   -H ≤ ... ≤ z ≤ ξp ≤ ... ≤ 0
    #
    # Therefore:
    #
    #   ξ ∈ [-H, z]     → ξ ≤ z and ξ < ξp   → f22
    #   ξ ∈ [z, ξp]     → ξ ≥ z and ξ < ξp   → f12
    #   ξ ∈ [ξp, 0]     → ξ ≥ z and ξ ≥ ξp   → f11
    # ------------------------------------------------------------------
    println("  integrate_Guv_y_Gb_fast: Case A, z ≤ ξp")

    I_A_22 = integrate_terms_fast(
        f22,
        (ξ, -H, z);
        simplify_input=false,
        compress_every=compress_every,
        print_each_term=print_each_term,
    )

    println("    Computed I_A_22 = ∫_{-H}^{z} f22 dξ")

    I_A_12 = integrate_terms_fast(
        f12,
        (ξ, z, ξp);
        simplify_input=false,
        compress_every=compress_every,
        print_each_term=print_each_term,
    )

    println("    Computed I_A_12 = ∫_{z}^{ξp} f12 dξ")

    I_A_11 = integrate_terms_fast(
        f11,
        (ξ, ξp, 0);
        simplify_input=false,
        compress_every=compress_every,
        print_each_term=print_each_term,
    )

    println("    Computed I_A_11 = ∫_{ξp}^{0} f11 dξ")

    println("    Simplifying Case A Guv_y_b result...")
    I_A = medium_simplify(I_A_22 + I_A_12 + I_A_11)

    # ------------------------------------------------------------------
    # Case B: ξp < z
    #
    # Ordering:
    #
    #   -H ≤ ... ≤ ξp < z ≤ ... ≤ 0
    #
    # Therefore:
    #
    #   ξ ∈ [-H, ξp]    → ξ < ξp and ξ ≤ z   → f22
    #   ξ ∈ [ξp, z]     → ξ ≥ ξp and ξ ≤ z   → f21
    #   ξ ∈ [z, 0]      → ξ ≥ ξp and ξ ≥ z   → f11
    # ------------------------------------------------------------------
    println("  integrate_Guv_y_Gb_fast: Case B, ξp < z")

    I_B_22 = integrate_terms_fast(
        f22,
        (ξ, -H, ξp);
        simplify_input=false,
        compress_every=compress_every,
        print_each_term=print_each_term,
    )

    println("    Computed I_B_22 = ∫_{-H}^{ξp} f22 dξ")

    I_B_21 = integrate_terms_fast(
        f21,
        (ξ, ξp, z);
        simplify_input=false,
        compress_every=compress_every,
        print_each_term=print_each_term,
    )

    println("    Computed I_B_21 = ∫_{ξp}^{z} f21 dξ")

    I_B_11 = integrate_terms_fast(
        f11,
        (ξ, z, 0);
        simplify_input=false,
        compress_every=compress_every,
        print_each_term=print_each_term,
    )

    println("    Computed I_B_11 = ∫_{z}^{0} f11 dξ")

    println("    Simplifying Case B Guv_y_b result...")
    I_B = medium_simplify(I_B_22 + I_B_21 + I_B_11)

    # ------------------------------------------------------------------
    # Multiply by dφ/dy after integration.
    # ------------------------------------------------------------------
    println("  integrate_Guv_y_Gb_fast: Multiplying by φ_derivative.")

    I_A_y = φ_derivative * I_A
    I_B_y = φ_derivative * I_B

    # ------------------------------------------------------------------
    # Final staged simplification of each branch separately.
    # ------------------------------------------------------------------
    println("  integrate_Guv_y_Gb_fast: Simplifying final branches.")

    I_A_y = sympy.together(I_A_y)
    I_A_y = sympy.cancel(I_A_y)
    I_A_y = sympy.factor_terms(I_A_y)
    I_A_y = sympy.powsimp(I_A_y, force=false)

    I_B_y = sympy.together(I_B_y)
    I_B_y = sympy.cancel(I_B_y)
    I_B_y = sympy.factor_terms(I_B_y)
    I_B_y = sympy.powsimp(I_B_y, force=false)

    # ------------------------------------------------------------------
    # Final Piecewise result
    # ------------------------------------------------------------------
    println("  integrate_Guv_y_Gb_fast: Building Piecewise result.")

    Guv_y_b = sympy.Piecewise(
        (I_A_y, sympy.Le(z, ξp)),
        (I_B_y, sympy.Gt(z, ξp))
    )

    println("  integrate_Guv_y_Gb_fast: Done.")

    return Guv_y_b
end

In [ ]:
println()
println("1.7 Symbolic computation of Guv_y_b...")

@time begin
    empty!(integral_cache)
    empty!(dep_integral_cache)

    Guv_y_b = integrate_Guv_y_Gb_fast(
        Guv_y,
        inner_Gb_integral;
        print_each_term=false,
        compress_every=10_000,
    )
end

println("1.7 Done.")

#### Convert `sympy` objects to `Symbolics` and save them to disk

In [ ]:
# Safe access to SymPy boolean properties such as is_Add, is_Mul, is_Pow
function sympy_boolprop(expr, prop::Symbol)
    try
        return Bool(getproperty(expr, prop))
    catch
        return false
    end
end

# True only if expr itself is a Piecewise
is_sympy_piecewise(expr) = startswith(string(expr), "Piecewise(")

# Convert a SymPy Piecewise with 2 branches using an externally supplied Symbolics condition
function sympy_piecewise_to_symbolics(P, condsym)
    branches = P.args
    @assert length(branches) == 2 "Expected exactly 2 branches in Piecewise"

    b1 = branches[1]
    b2 = branches[2]

    # Each branch is a SymPy Tuple-like object:
    expr1_sympy = b1.args[1]
    expr2_sympy = b2.args[1]

    expr1 = sympy_to_symbolics_with_piecewise(expr1_sympy, condsym)
    expr2 = sympy_to_symbolics_with_piecewise(expr2_sympy, condsym)

    return ifelse(condsym, expr1, expr2)
end

# Recursive converter
function sympy_to_symbolics_with_piecewise(expr, condsym)

    # Case 1: expr itself is Piecewise
    if is_sympy_piecewise(expr)
        return sympy_piecewise_to_symbolics(expr, condsym)
    end

    # Case 2: expr is a sum
    if sympy_boolprop(expr, :is_Add)
        out = 0
        for a in expr.args
            out += sympy_to_symbolics_with_piecewise(a, condsym)
        end
        return out
    end

    # Case 3: expr is a product
    if sympy_boolprop(expr, :is_Mul)
        out = 1
        for a in expr.args
            out *= sympy_to_symbolics_with_piecewise(a, condsym)
        end
        return out
    end

    # Case 4: expr is a power
    if sympy_boolprop(expr, :is_Pow)
        base = sympy_to_symbolics_with_piecewise(expr.args[1], condsym)
        expo = sympy_to_symbolics_with_piecewise(expr.args[2], condsym)
        return base ^ expo
    end

    # Important safety check:
    # If we get here and the expression still contains Piecewise,
    # we must not send it to sympy_to_symbolics.
    if occursin("Piecewise", string(expr))
        error("""
        Expression contains Piecewise but was not decomposed.

        expr = $expr

        args = $(expr.args)
        """)
    end

    # Case 5: ordinary SymPy expression
    return sympy_to_symbolics(expr, [])
end

In [ ]:
println()
println("1.8 Expressions conversion to Symbolics and save to file...")

@variables z ξ ξp
@time symbolics_Gb                 = sympy_to_symbolics_with_piecewise(Gb, z <= ξ)
@time symbolics_Guv                = sympy_to_symbolics_with_piecewise(Guv, z <= ξ)
@time symbolics_Guv_y              = sympy_to_symbolics_with_piecewise(Guv_y, z <= ξ)

@time symbolics_inner_Gb_integral  = sympy_to_symbolics_with_piecewise(inner_Gb_integral, ξ >= ξp)
@time symbolics_inner_Gb2_integral = sympy_to_symbolics(inner_Gb2_integral, [])

@time symbolics_Guvb               = sympy_to_symbolics_with_piecewise(Guvb, z <= ξp)
@time symbolics_Guvb_xi            = sympy_to_symbolics_with_piecewise(Guvb_xi, z <= ξp)
@time symbolics_Guv_y_b_xi         = sympy_to_symbolics(Guv_y_b_xi, [])
@time symbolics_Guv_y_b            = sympy_to_symbolics_with_piecewise(Guv_y_b, z <= ξp)

@save "symexpr.jld2" symbolics_Guvb symbolics_Gb symbolics_Guv symbolics_inner_Gb_integral symbolics_Guvb_xi symbolics_inner_Gb2_integral symbolics_Guv_y symbolics_Guv_y_b_xi symbolics_Guv_y_b

println("1.8 Done and saved to: symexpr.jld2")
println()
println("Check these expressions by running SymbolicsTesting.jl in a fresh Julia REPL.")
println("Plot the Guvb expression by running Plot_Guvb.ipynb.")